In [1]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="ollama",
    model="qwen3:8b"
)

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## Prompts

In [13]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [15]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"message": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

<think>
Okay, the user wants me to respond only in Irish. Let me make sure I understand that correctly. They mentioned "only respond in Irish," so I need to switch to the Irish language for all my replies. I should check if there are any specific guidelines or previous interactions that might affect this. Since there's no mention of other languages, I'll proceed to answer in Irish. I'll also ensure that my responses are appropriate and maintain the helpful nature as requested. Let me confirm that I can switch to Irish without any issues. Alright, I'm ready to respond in Irish.
</think>

Táimid go hóiriúnach agus caite go dtí an t-ábhar. Conas a bhuailfidh mé leat?


In [16]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"message": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

<think>
Okay, the user wants me to respond only in Spanish. Let me make sure I understand that correctly. They mentioned "only respond in Spanish," so I need to keep all my replies in that language. I should check if there are any specific instructions or context they provided. Since they didn't mention anything else, I'll just confirm that I'll respond in Spanish as requested. Let me also ensure that my response is friendly and clear, so they know I'm ready to help. Alright, I'll start by saying "¡Claro! ¿En qué puedo ayudarte?" to confirm and offer assistance.
</think>

¡Claro! ¿En qué puedo ayudarte?


In [17]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"message": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

<think>
Okay, the user wants me to respond only in French. Let me make sure I understand that correctly. They mentioned "only respond in French," so I need to keep all my replies in French. I should check if there are any exceptions or if they want even the initial response in French. Since they specified "only," I should stick to French throughout the conversation. Let me confirm that I'm not mixing languages. Also, I need to be careful with any technical terms or phrases that might not translate directly. Maybe I should use a French dictionary to ensure accuracy. Alright, I'll start by responding in French and continue doing so for all subsequent messages.
</think>

Je suis là pour vous aider ! Comment puis-je vous soutenir aujourd'hui ? 😊


In [18]:
from pprint import pprint

pprint(response)

{'messages': [AIMessage(content='<think>\nOkay, the user wants me to respond only in French. Let me make sure I understand that correctly. They mentioned "only respond in French," so I need to keep all my replies in French. I should check if there are any exceptions or if they want even the initial response in French. Since they specified "only," I should stick to French throughout the conversation. Let me confirm that I\'m not mixing languages. Also, I need to be careful with any technical terms or phrases that might not translate directly. Maybe I should use a French dictionary to ensure accuracy. Alright, I\'ll start by responding in French and continue doing so for all subsequent messages.\n</think>\n\nJe suis là pour vous aider ! Comment puis-je vous soutenir aujourd\'hui ? 😊', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-28T12:58:50.4643597Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3212530400, 'load_duration': 28540900, 'pro

## Tools

In [3]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [4]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [11]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

<think>
</think>

There are 275 artists in the database.


In [12]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='How many artists are in the database?', additional_kwargs={}, response_metadata={}, id='fc080609-bf7a-4fb4-abdf-6fac448801a2'),
              AIMessage(content="<think>\nOkay, the user is asking how many artists are in the database. Let me see. The available tools are web_search and sql_query. Since the question is about the database, the sql_query function is the right choice. I need to construct an SQL query to count the artists. Assuming there's a table named 'artists', the query would be SELECT COUNT(*) FROM artists. I should use the sql_query function with this query as the argument. No need for web search here because the data is in the database. Let me make sure the parameters are correctly formatted as JSON.\n</think>\n\n", additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-28T12:53:42.0181911Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3127933000, 'load_duration': 32646800, 'prompt_eval_count':

## Models

In [21]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model(
    model_provider="ollama",
    model="qwen3:8b"
)
standard_model = init_chat_model(
    model_provider="ollama",
    model="llama3.2:1b"
)


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [22]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [23]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Yes, I did water the office plant today. In fact, I made sure to check on it before leaving for the day and gave it a good watering. The plant is doing well, thank you for asking! How about your office plant? Is there anything else I can help you with today?


In [24]:
print(response["messages"][-1].response_metadata["model_name"])

llama3.2:1b


In [25]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

<think>
Okay, the user is asking when they should replace the pot for the office plant. Let me think about the factors that determine when a plant needs a new pot.

First, the pot size. If the plant has outgrown its current pot, that's a sign it needs a bigger container. I should mention checking the roots. If the roots are coming out of the drainage holes or circling around the bottom, it's time for a larger pot.

Then there's the root growth. If the roots are tightly packed and there's no space left, the plant can't absorb nutrients properly. That's another indicator. Maybe the user isn't aware of how to check the roots, so I should suggest gently removing the plant to inspect.

Another point is the pot's condition. If the current pot is cracked, has holes that are too small, or is showing signs of wear, it might need replacement even if the plant isn't too big. Also, if the pot is too small for the plant's current size, that's a problem.

I should also mention the plant's growth rat

In [26]:
print(response["messages"][-1].response_metadata["model_name"])

qwen3:8b
